# OASIS Alzheimer — 3-class training (Kaggle)

Subject-level, leak-free training skeleton for the 3-class task
(`no_dementia` / `very_mild` / `mild_moderate`). Mirrors `research.md`.

**Runtime:** Kaggle Notebook, GPU (T4/P100) ON. Dataset: `ninadaithal/imagesoasis`.

Pipeline: rebuild the subject-level split on Kaggle (same seed → same folds) →
central-slice filter → two-stage transfer learning → **subject-level** metrics.


## 0. GPU sanity check (run this first)

Run this cell alone before anything else. If it reports `CPU only`, stop: turn the
accelerator on (sidebar -> Accelerator -> GPU T4 x2), let the session restart, then
Run All. Training this on CPU takes hours instead of minutes.


In [ ]:
import torch

print("torch      :", torch.__version__)
print("cuda avail :", torch.cuda.is_available())
print("device     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
if not torch.cuda.is_available():
    print("\n>>> NO GPU. Sidebar -> Accelerator -> GPU T4 x2, accept the restart, then Run All.")
    print(">>> Also check your weekly GPU quota -- if it is exhausted Kaggle silently gives you CPU.")


## 1. Config

In [ ]:
import os, re, random, glob
from pathlib import Path
import numpy as np, pandas as pd
# claude --resume ef095864-a42b-410e-884c-f717ed1e6466
def resolve_data_root(preferred):
    """Return `preferred` if it has class folders; otherwise auto-find the OASIS
    `Data` dir anywhere under /kaggle/input (handles repo uploads that nest it
    under datasets/Data/, and differing dataset slugs)."""
    if Path(preferred).is_dir():
        return preferred
    for cand in glob.glob("/kaggle/input/**/Data", recursive=True):
        if (Path(cand) / "Non Demented").is_dir():
            return cand
    return preferred  # fall through -> scan() returns empty -> guard fires with a clear message

CFG = dict(
    # Public dataset default; auto-resolved below for repo uploads (e.g. topotisarkar/oasis-dataset).
    data_root   = resolve_data_root("/kaggle/input/imagesoasis/Data"),
    seed        = 42,
    test_frac   = 0.20,
    n_folds     = 5,
    # All five folds, always. One fold holds 3-4 mild_moderate subjects, so a single
    # subject flipping moves macro-F1 by ~0.05 -- a one-fold number is noise, not a result.
    run_folds   = [0, 1, 2, 3, 4],
    # Slice window. 128..132 (5 slices/subject) trains fast but gives ~1.1k images,
    # far too few for an 11M-param net -> it memorises subjects. 124..136 is ~2.6x
    # the data for ~2.6x the epoch time, and is the single biggest anti-overfit lever.
    central_lo  = 124,
    central_hi  = 136,
    # 2.5D input: stack slices (s-gap, s, s+gap) as the three RGB channels instead of
    # copying one grayscale slice three times. The old loader put the same image in all
    # three channels -- two thirds of the input was redundant. Three adjacent slices
    # give the net real anatomical context for zero extra parameters.
    slice_25d   = True,
    slice_gap   = 2,            # slices between channels; False -> old grayscale-x3
    backbone    = "resnet18",   # "resnet18" (recommended baseline) | "resnet50" | "efficientnet_b0"
    img_size    = 224,
    batch_size  = 32,
    stage1_epochs = 3,          # frozen backbone, train head only
    stage2_epochs = 12,         # partial fine-tune (see freeze_until)
    # Stage 2 only unfreezes the LAST blocks. Fine-tuning all 11.18M params on ~3k
    # images is what collapsed train loss to 0.16 while val F1 fell. Note that
    # resnet18's parameters are back-loaded -- layer4 alone is 75% of the net -- so
    # freezing early blocks saves less than it looks like it should. Real but modest.
    #   "head"   -> linear probe, 1.5k trainable (strongest regulariser; same as stage 1)
    #   "layer4" -> train layer4 + head,   8.40M / 11.18M (75%)   <- default
    #   "layer3" -> train layer3 onward,  10.49M / 11.18M (94%)
    #   None     -> unfreeze everything (the old, overfitting behaviour)
    # MEASURED: on the cached features a plain frozen-backbone linear probe scored
    # subject macro-F1 0.58-0.61 over 5 folds while the layer4 fine-tune scored 0.375
    # on fold 0 (probe on the same fold: 0.561). Fine-tuning 8.4M params on ~11k
    # slices from ~220 subjects destroys the pretrained features. Start frozen; only
    # move to "layer4" if it beats the probe on ALL five folds.
    freeze_until = "head",
    lr_head      = 1e-3,        # stage 1: head on a frozen backbone
    lr_head_ft   = 2e-4,        # stage 2: head keeps a higher LR than the backbone
    lr_backbone  = 2e-5,        # stage 2: unfrozen backbone blocks
    weight_decay = 1e-2,        # AdamW decoupled WD; 1e-4 was far too weak for this data size
    label_smoothing = 0.05,     # stops the net driving train logits to certainty
    dropout      = 0.5,
    # Augmentation strength. Everything here preserves brain-to-frame SCALE -- see the
    # build_tf docstring for why that is not negotiable on this task.
    aug_degrees   = 8,
    aug_translate = 0.02,
    aug_jitter    = 0.15,
    patience     = 4,           # early stop on INNER-val macro-F1
    # Nested early stopping. Choosing the stopping epoch on the fold you then report
    # is selection on the test set: with labels randomly permuted, the diagnostic
    # battery still scored macro-F1 0.394 that way against a 0.289 baseline, so ~0.10
    # of any "best epoch" headline is manufactured. The stopping signal now comes from
    # a held-out slice of the TRAINING subjects; the outer fold is scored once, at the
    # end, by a model that never influenced when training stopped.
    nested_early_stop = True,
    inner_frac        = 0.2,    # share of training SUBJECTS reserved for stopping
    report_peek_bias  = True,   # also track the old best-on-outer number, to size the gap
    # Cached-feature linear probe (section 5b). A frozen backbone produces the same
    # feature vector every epoch, so recomputing it is waste: extract once, cache, then
    # train only the head on the cached vectors. Seconds per experiment instead of
    # minutes, which is what makes a 5-fold sweep affordable. stage1_epochs=3 left this
    # head nowhere near converged, so the probe is the honest linear baseline.
    probe_views    = 4,         # augmented feature views cached per image (0 = none)
    probe_epochs   = 30,
    probe_lr       = 1e-3,
    probe_patience = 8,
    # Section 5c/5d: scikit-learn reference probe and hierarchical cascade. These fit in
    # closed form on the cached features, so a whole 5-fold sweep costs seconds and
    # gives a trustworthy floor to judge every neural change against.
    sk_c_grid      = [3e-4, 1e-3, 3e-3, 1e-2, 1e-1, 1.0],   # inverse L2 strength
    cascade_agg    = "feature_mean",
    cascade_c_binary = 1.0,
    cascade_c_fine   = 1e-1,
    num_workers  = 2,
    use_sampler  = True,        # WeightedRandomSampler for class balance (image-appropriate; see step 6).
    use_amp      = True,        # mixed precision -> ~2x faster on T4/P100 GPUs.
    cache_split  = True,        # cache the scanned/split table so reruns skip the slow file scan.
)
CLASS_NAMES = ["no_dementia", "very_mild", "mild_moderate"]
print("data_root ->", CFG["data_root"])

def set_seed(s):
    random.seed(s); np.random.seed(s)
    import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed(CFG["seed"])

## 2. Rebuild the split on Kaggle

Deterministic and subject-grouped, so it reproduces the local folds exactly
(same seed). Moderate → Mild merge, central-slice filter applied here.


In [ ]:
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from PIL import Image

FOLDER_TO_LABEL = {           # raw folder -> 3-class label (Moderate folded into Mild)
    "Non Demented": 0, "Very mild Dementia": 1,
    "Mild Dementia": 2, "Moderate Dementia": 2,
}
SUBJECT_RE = re.compile(r"(OAS1_\d+)", re.IGNORECASE)
SLICE_RE   = re.compile(r"_(\d+)\.(?:jpg|jpeg|png)$", re.IGNORECASE)

def scan(data_root):
    rows = []
    for folder, label in FOLDER_TO_LABEL.items():
        d = Path(data_root) / folder
        for p in sorted(d.glob("*")):
            if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}: continue
            sm, im = SUBJECT_RE.search(p.name), SLICE_RE.search(p.name)
            if not sm: continue
            rows.append(dict(filepath=str(p), subject_id=sm.group(1).upper(),
                             label=label, slice_idx=int(im.group(1)) if im else -1))
    return pd.DataFrame(rows)

def drop_unreadable(df):
    """Filter out images PIL can't decode (truncated/corrupt uploads). Reopens and
    fully loads each file, since verify() alone misses some truncation."""
    ok = []
    for p in df["filepath"]:
        try:
            with Image.open(p) as im:
                im.load()
            ok.append(True)
        except Exception:
            ok.append(False)
    ok = pd.Series(ok, index=df.index)
    bad = int((~ok).sum())
    if bad:
        print(f"WARNING: dropped {bad} unreadable image(s) of {len(df)} "
              f"({bad/len(df):.1%}). If this is most of the set, the Kaggle "
              f"upload is corrupt -> re-upload the dataset.")
    return df[ok].copy()

def assign_folds(slices, test_frac, n_folds, seed):
    subj = (slices.groupby("subject_id")["label"].first().reset_index()
            .sort_values("subject_id").reset_index(drop=True))
    y = subj["label"].to_numpy()
    dev_idx, _ = next(StratifiedShuffleSplit(1, test_size=test_frac,
                                             random_state=seed).split(subj, y))
    subj["fold"] = -1
    dev = subj.iloc[dev_idx].reset_index()
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=seed)
    for f, (_, val) in enumerate(skf.split(dev, dev["label"])):
        subj.loc[dev.loc[val, "index"].to_numpy(), "fold"] = f
    return slices.merge(subj[["subject_id", "fold"]], on="subject_id", how="left")

def series_id(paths):
    """Path minus the trailing `_<slice>.jpg`, i.e. one scan run (`OAS1_0028_MR1_mpr-2`).
    Subjects have up to 4 mpr runs plus MR2 sessions, so (subject_id, slice_idx) is NOT
    unique -- keying on it would pair a slice with a "neighbour" from a different,
    unregistered scan."""
    return [SLICE_RE.sub("", p) for p in paths]

def add_neighbour_paths(central, slices, gap):
    """Attach the paths of the slices `gap` above/below each central slice (same scan
    run) so the loader can stack them as RGB channels. The lookup is built from the
    *full* scan, not the central window, so slices at the window edge still get real
    neighbours. Anything genuinely missing falls back to the centre slice."""
    lut = dict(zip(zip(series_id(slices.filepath), slices.slice_idx), slices.filepath))
    ser = series_id(central.filepath)
    for col, d in (("f_lo", -gap), ("f_hi", +gap)):
        central[col] = [lut.get((k, i + d), fp) for k, i, fp
                        in zip(ser, central.slice_idx, central.filepath)]
    edge = int((central.f_lo == central.filepath).sum()
               + (central.f_hi == central.filepath).sum())
    print(f"2.5D neighbours: {edge:,} of {2*len(central):,} fell back to the centre slice")
    return central

def build_central():
    """Scan files -> assign folds -> keep central slices -> attach 2.5D neighbours ->
    drop unreadable ones."""
    slices = scan(CFG["data_root"])
    assert len(slices), (
        f"No images found under {CFG['data_root']!r}. Check the Add Input path / data_root. "
        f"Folders present under /kaggle/input: {glob.glob('/kaggle/input/*')}")
    slices = assign_folds(slices, CFG["test_frac"], CFG["n_folds"], CFG["seed"])
    central = slices[slices.slice_idx.between(CFG["central_lo"], CFG["central_hi"])].copy()
    if CFG["slice_25d"]:
        central = add_neighbour_paths(central, slices, CFG["slice_gap"])
    central = drop_unreadable(central)          # verify only the slices we actually train on
    assert len(central), "All central-slice images were unreadable -> re-upload the dataset."
    return central

# Cache: scanning tens of thousands of files is slow, so save the split table and
# reuse it on later runs. The cache key includes the slice window, so changing
# central_lo/central_hi rebuilds automatically. Note: /kaggle/working starts empty
# each session unless you save the notebook output / enable persistence.
CACHE = Path(f"/kaggle/working/central_{CFG['central_lo']}_{CFG['central_hi']}"
             f"_g{CFG['slice_gap'] if CFG['slice_25d'] else 0}.parquet")
if CFG["cache_split"] and CACHE.exists():
    central = pd.read_parquet(CACHE)
    print(f"loaded cached split from {CACHE.name}")
else:
    central = build_central()
    if CFG["cache_split"]:
        central.to_parquet(CACHE)
        print(f"saved split cache -> {CACHE.name}")

print(f"central {CFG['central_lo']}-{CFG['central_hi']} (readable): {len(central):,}")
print(central.groupby("fold")["subject_id"].nunique())

## 3. Dataset + transforms

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True   # tolerate end-truncated JPEGs

IMAGENET_MEAN, IMAGENET_STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

def build_tf(train):
    """Train-time augmentation, constrained to transforms that PRESERVE SCALE.

    Atrophy presents as ventricle and brain size relative to the skull, so anything
    that rescales the image randomises the exact quantity we are classifying on. The
    previous recipe used RandomResizedCrop(scale=(0.75, 1.0)) plus RandomErasing;
    measured on the cached features, a linear probe trained on those augmented views
    scored subject macro-F1 0.35 against 0.58 for the same images unaugmented. That
    is signal destruction, not regularisation.

    Rotation, small translation, horizontal flip (brains are near-symmetric) and
    photometric jitter all leave the size ratio intact. Val/test stay deterministic.

    After changing anything here, re-run `sk_probe_sweep(F, use_aug=True)` in section
    5c and check the score does not fall below the deterministic sweep.
    """
    base = [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    resize = transforms.Resize((CFG["img_size"], CFG["img_size"]))
    if not train:
        return transforms.Compose([resize] + base)
    return transforms.Compose([
        resize,
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomAffine(degrees=CFG["aug_degrees"],
                                translate=(CFG["aug_translate"], CFG["aug_translate"])),
        transforms.ColorJitter(brightness=CFG["aug_jitter"], contrast=CFG["aug_jitter"]),
    ] + base)

class OasisDS(Dataset):
    def __init__(self, df, train, retries=4):
        self.df = df.reset_index(drop=True); self.tf = build_tf(train)
        self.retries = retries
        # A split table cached before slice_25d existed has no neighbour columns.
        self.use_25d = bool(CFG["slice_25d"]) and {"f_lo", "f_hi"} <= set(self.df.columns)
        if CFG["slice_25d"] and not self.use_25d:
            print("WARN: slice_25d=True but the split table has no f_lo/f_hi columns -> "
                  "falling back to grayscale-x3. Delete the central_*.parquet cache and "
                  "re-run section 2.")

    def __len__(self): return len(self.df)

    def _load(self, i):
        r = self.df.iloc[i]
        if self.use_25d:
            # The augmentation is applied to the merged image, so the three slices stay
            # registered to each other.
            chans = tuple(Image.open(p).convert("L") for p in (r.f_lo, r.filepath, r.f_hi))
        else:
            g = Image.open(r.filepath).convert("L")
            chans = (g, g, g)
        return self.tf(Image.merge("RGB", chans)), int(r.label), r.subject_id

    def __getitem__(self, i):
        # Flaky GitHub-import mounts occasionally return partial reads; retry the
        # same file, then fall back to the next index so one bad read never
        # kills the epoch.
        for attempt in range(self.retries):
            try:
                return self._load((i + attempt) % len(self.df))
            except Exception as e:
                if attempt == self.retries - 1:
                    print(f"WARN: skipping unreadable {self.df.iloc[i].filepath} ({e})")
        return self._load((i + self.retries) % len(self.df))

## 4. Model factory (backbone-agnostic)

In [ ]:
import torch.nn as nn
from torchvision import models

HEAD_ATTR = {"resnet18": "fc", "resnet50": "fc", "efficientnet_b0": "classifier"}

# Blocks in order, shallow -> deep. Stage 2 unfreezes from `freeze_until` onward.
BLOCK_ORDER = {
    "resnet18": ["conv1", "bn1", "layer1", "layer2", "layer3", "layer4"],
    "resnet50": ["conv1", "bn1", "layer1", "layer2", "layer3", "layer4"],
    # efficientnet_b0 has one `features` Sequential; index it by stage.
    "efficientnet_b0": [f"features.{i}" for i in range(9)],
}

def build_model(backbone, n_classes=3, dropout=None):
    dropout = CFG["dropout"] if dropout is None else dropout
    if backbone == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        feat = m.fc.in_features
    elif backbone == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        feat = m.fc.in_features
    elif backbone == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        feat = m.classifier[1].in_features
    else:
        raise ValueError(backbone)
    setattr(m, HEAD_ATTR[backbone], nn.Sequential(nn.Dropout(dropout), nn.Linear(feat, n_classes)))
    return m

def set_trainable(model, backbone, freeze_until):
    """Freeze everything shallower than `freeze_until`; train that block onward + head.

    freeze_until="head" -> backbone fully frozen (stage 1)
    freeze_until=None   -> nothing frozen (full fine-tune)
    freeze_until="layer3" -> conv1/bn1/layer1/layer2 frozen, layer3+layer4+head trained
    """
    head = HEAD_ATTR[backbone]
    order = BLOCK_ORDER[backbone]
    if freeze_until == "head":
        frozen = set(order)
    elif freeze_until is None:
        frozen = set()
    else:
        assert freeze_until in order, f"{freeze_until!r} not in {order}"
        frozen = set(order[:order.index(freeze_until)])

    for name, p in model.named_parameters():
        if name.startswith(head):
            p.requires_grad = True
        else:
            p.requires_grad = not any(name.startswith(b + ".") or name == b for b in frozen)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"  trainable: {n_tr/1e6:.2f}M / {n_all/1e6:.2f}M params (freeze_until={freeze_until})")

def freeze_frozen_bn(model):
    """Put BatchNorm layers whose params are frozen into eval mode.

    model.train() would otherwise keep updating their running mean/var even though
    requires_grad=False — the frozen ImageNet statistics drift toward this tiny
    dataset, which is a quiet source of overfitting during partial fine-tuning.
    """
    for mod in model.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm):
            if not any(p.requires_grad for p in mod.parameters(recurse=False)):
                mod.eval()

def param_groups(model, backbone, lr_backbone, lr_head):
    """Discriminative LRs: pretrained backbone blocks move slowly, the new head faster."""
    head = HEAD_ATTR[backbone]
    bb = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith(head)]
    hd = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith(head)]
    groups = []
    if bb: groups.append({"params": bb, "lr": lr_backbone})
    if hd: groups.append({"params": hd, "lr": lr_head})
    return groups

## 5. Train / eval helpers (subject-level metrics)

In [ ]:
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)
from sklearn.model_selection import StratifiedShuffleSplit

device = "cuda" if torch.cuda.is_available() else "cpu"
AMP = CFG["use_amp"] and device == "cuda"   # mixed precision only helps (and is safe) on GPU

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    train = optimizer is not None
    if train:
        model.train()
        freeze_frozen_bn(model)   # keep frozen blocks' BN running stats pinned
    else:
        model.eval()
    total = 0.0
    with torch.set_grad_enabled(train):
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            if train: optimizer.zero_grad()
            # autocast runs the forward pass in fast float16 where it's safe.
            with torch.autocast(device_type="cuda", enabled=AMP):
                out = model(x); loss = criterion(out, y)
            if train:
                if scaler is not None:
                    # GradScaler prevents float16 gradients from underflowing to zero.
                    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                else:
                    loss.backward(); optimizer.step()
            total += loss.item() * x.size(0)
    return total / len(loader.dataset)

@torch.no_grad()
def subject_eval(model, loader, criterion=None):
    """Subject-level metrics. Pass `criterion` to also get val loss — watching
    train loss alone hides overfitting until the F1 has already collapsed."""
    model.eval()
    probs, subs, ys, loss_sum = [], [], [], 0.0
    for x, y, s in loader:
        x = x.to(device)
        logits = model(x)
        if criterion is not None:
            loss_sum += criterion(logits, y.to(device)).item() * x.size(0)
        probs.append(torch.softmax(logits, 1).cpu().numpy()); subs.extend(s); ys.extend(y.numpy())
    probs = np.concatenate(probs)
    df = pd.DataFrame(probs, columns=CLASS_NAMES); df["subject_id"] = subs; df["y"] = ys
    g = df.groupby("subject_id")
    pred = g[CLASS_NAMES].mean().values.argmax(1)      # mean-prob aggregation
    true = g["y"].first().values
    return dict(macro_f1=f1_score(true, pred, average="macro"),
                bal_acc=balanced_accuracy_score(true, pred),
                val_loss=(loss_sum / len(loader.dataset)) if criterion is not None else float("nan"),
                report=classification_report(true, pred, target_names=CLASS_NAMES, zero_division=0),
                cm=confusion_matrix(true, pred), true=true, pred=pred)

def inner_val_subjects(subject_ids, labels, frac=None, seed=None):
    """Hold out a stratified slice of the TRAINING subjects to early-stop on.

    Grouped by subject like the outer split, for the same reason: a subject whose
    slices sit on both sides of the line makes the stopping signal optimistic too.

    Returns a set of subject_ids. With inner_frac=0.2 the stopping set holds roughly
    44 subjects, of which only ~4 are mild_moderate -- noisy, but noise in the
    stopping signal costs you a slightly wrong epoch, whereas peeking at the outer
    fold costs you a wrong headline.
    """
    frac = CFG["inner_frac"] if frac is None else frac
    seed = CFG["seed"] if seed is None else seed
    s = (pd.DataFrame({"subject_id": np.asarray(subject_ids), "label": np.asarray(labels)})
         .drop_duplicates("subject_id").sort_values("subject_id").reset_index(drop=True))
    _, va = next(StratifiedShuffleSplit(1, test_size=frac, random_state=seed)
                 .split(s, s["label"]))
    return set(s.loc[va, "subject_id"])

def summarise(results, title):
    """Mean +/- std over folds, with the peek gap when it was tracked."""
    f1 = [results[f]["macro_f1"] for f in results]
    ba = [results[f]["bal_acc"] for f in results]
    print(f"\n{title}")
    print(f"  macro-F1 (honest, outer fold) : {np.mean(f1):.4f} +/- {np.std(f1):.4f}   "
          f"{[round(v, 3) for v in f1]}")
    print(f"  bal-acc                       : {np.mean(ba):.4f} +/- {np.std(ba):.4f}")
    peek = [results[f].get("peek_macro_f1") for f in results]
    if all(p is not None for p in peek):
        print(f"  macro-F1 if stopped ON the outer fold (INVALID, for reference only): "
              f"{np.mean(peek):.4f}")
        print(f"  selection bias avoided        : {np.mean(peek) - np.mean(f1):+.4f}")
    print("  majority baseline macro-F1 0.289 / bal-acc 0.333")

def class_weights(df):
    counts = df["label"].value_counts().sort_index().values
    w = counts.sum() / (len(counts) * counts)
    return torch.tensor(w, dtype=torch.float32, device=device)

## 5b. Cached-feature linear probe (fast baseline)

The backbone is frozen in stage 1, so every image has one fixed feature vector —
recomputing it each epoch is pure waste. Extract the features **once**, cache them, and
train only the head on those vectors: a 5-fold probe then costs seconds instead of
minutes, which is what makes sweeping LR / weight decay / dropout / slice window
practical.

Two details that keep this a fair baseline rather than a shortcut:

- **Augmentation is baked in.** A cached vector can't be re-augmented, so `probe_views`
  augmented passes are extracted per image alongside one deterministic pass. Training
  uses both; validation uses the deterministic view only.
- **Standardisation is fit on training rows only**, per fold — no leakage across the
  split.

`stage1_epochs = 3` (section 6) left this head far from converged, so treat the number
below, not that one, as your linear-probe baseline. Test subjects (`fold == -1`) are
never extracted.


In [ ]:
from torch.utils.data import TensorDataset, WeightedRandomSampler

def feature_extractor(backbone):
    """The pretrained backbone with its classifier replaced by a pass-through, so a
    forward pass returns the pooled vector the head would have consumed."""
    m = build_model(backbone)
    setattr(m, HEAD_ATTR[backbone], nn.Identity())
    return m.eval().to(device)

@torch.no_grad()
def extract(df, model, train_tf, views):
    """Run the frozen backbone over `df` `views` times and stack the passes. Rows come
    back in df order, repeated once per view (shuffle=False), so a fold mask over df
    tiles to match."""
    if views == 0:
        return torch.empty(0), torch.empty(0, dtype=torch.long), np.array([], dtype=object)
    feats, ys, subs = [], [], []
    for v in range(views):
        set_seed(CFG["seed"] + v)          # each view a different but reproducible draw
        dl = DataLoader(OasisDS(df, train_tf), CFG["batch_size"], shuffle=False,
                        num_workers=CFG["num_workers"], pin_memory=True)
        for x, y, s in dl:
            with torch.autocast(device_type="cuda", enabled=AMP):
                f = model(x.to(device))
            feats.append(f.float().cpu()); ys.append(y); subs.extend(s)
    return torch.cat(feats), torch.cat(ys), np.array(subs, dtype=object)

def cached_features(central):
    """Extract (or reload) frozen features for every dev image. The cache tag covers
    everything that changes a feature vector, so any config change rebuilds."""
    dev = central[central.fold != -1].reset_index(drop=True)   # test set stays untouched
    gap = CFG["slice_gap"] if CFG["slice_25d"] else 0
    # The augmentation settings belong in the tag: X_aug changes when they change, and a
    # silently reused stale cache would make an augmentation experiment meaningless.
    aug = f"a{CFG['aug_degrees']}-{CFG['aug_translate']}-{CFG['aug_jitter']}"
    tag = (f"{CFG['backbone']}_{CFG['central_lo']}-{CFG['central_hi']}_g{gap}"
           f"_s{CFG['img_size']}_v{CFG['probe_views']}_{aug}_seed{CFG['seed']}")
    path = Path(f"/kaggle/working/feats_{tag}.pt")
    if path.exists():
        print(f"loaded cached features <- {path.name}")
        return torch.load(path, weights_only=False)

    model = feature_extractor(CFG["backbone"])
    print(f"extracting {CFG['backbone']} features for {len(dev):,} dev images "
          f"(1 deterministic + {CFG['probe_views']} augmented views)...")
    X_det, y_det, s_det = extract(dev, model, False, 1)
    X_aug, y_aug, s_aug = extract(dev, model, True,  CFG["probe_views"])
    del model; torch.cuda.empty_cache()

    fold_det = dev["fold"].to_numpy()
    F = dict(X_det=X_det, y_det=y_det, s_det=s_det, fold_det=torch.as_tensor(fold_det),
             X_aug=X_aug, y_aug=y_aug, s_aug=s_aug,
             fold_aug=torch.as_tensor(np.tile(fold_det, CFG["probe_views"])))
    torch.save(F, path)
    print(f"cached -> {path.name}  ({path.stat().st_size/1e6:.0f} MB, "
          f"{X_det.shape[1]}-d features)")
    return F

def aggregate_subject(probs, subs, true):
    """The same mean-probability subject aggregation `subject_eval` does, over arrays
    instead of a loader."""
    df = pd.DataFrame(probs, columns=CLASS_NAMES); df["subject_id"] = subs; df["y"] = true
    g = df.groupby("subject_id")
    pred = g[CLASS_NAMES].mean().values.argmax(1)
    t = g["y"].first().values
    return dict(macro_f1=f1_score(t, pred, average="macro"),
                bal_acc=balanced_accuracy_score(t, pred),
                report=classification_report(t, pred, target_names=CLASS_NAMES,
                                             zero_division=0),
                cm=confusion_matrix(t, pred), true=t, pred=pred)

def train_probe(F, fold, verbose=True):
    """Train the head on cached features; stop on an inner split of the training
    subjects; score the outer fold once with the selected weights."""
    fold_det, fold_aug = F["fold_det"].numpy(), F["fold_aug"].numpy()
    s_det, y_det = np.asarray(F["s_det"]), F["y_det"].numpy()
    has_aug = len(F["X_aug"]) > 0
    views = (len(F["X_aug"]) // len(F["X_det"])) if has_aug else 0

    dev_m, va_m = fold_det != fold, fold_det == fold
    aug_m = (fold_aug != fold) if has_aug else np.zeros(0, dtype=bool)
    if CFG["nested_early_stop"]:
        inner = inner_val_subjects(s_det[dev_m], y_det[dev_m])
        is_inner = np.isin(s_det, list(inner))
        es_m, tr_m = dev_m & is_inner, dev_m & ~is_inner
        if has_aug:
            aug_m = aug_m & ~np.tile(is_inner, views)   # stopping set stays unaugmented
    else:
        es_m, tr_m = va_m, dev_m

    tr_X, tr_y = F["X_det"][tr_m], F["y_det"][tr_m]
    if has_aug:
        tr_X = torch.cat([tr_X, F["X_aug"][aug_m]])
        tr_y = torch.cat([tr_y, F["y_aug"][aug_m]])

    # Standardise with training statistics only.
    mu, sd = tr_X.mean(0, keepdim=True), tr_X.std(0, keepdim=True).clamp_min(1e-6)
    norm = lambda X: ((X - mu) / sd).to(device)
    tr_X = norm(tr_X)
    es_X, es_y, es_s = norm(F["X_det"][es_m]), F["y_det"][es_m], s_det[es_m]
    va_X, va_y, va_s = norm(F["X_det"][va_m]), F["y_det"][va_m], s_det[va_m]

    counts = torch.bincount(tr_y, minlength=len(CLASS_NAMES)).float()
    cw = counts.sum() / (len(counts) * counts)

    ds = TensorDataset(tr_X, tr_y.to(device))
    if CFG["use_sampler"]:
        w = cw[tr_y]
        dl = DataLoader(ds, 256, sampler=WeightedRandomSampler(w, len(w), replacement=True))
    else:
        dl = DataLoader(ds, 256, shuffle=True)

    head = nn.Sequential(nn.Dropout(CFG["dropout"]),
                         nn.Linear(tr_X.shape[1], len(CLASS_NAMES))).to(device)
    crit = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"],
                               weight=None if CFG["use_sampler"] else cw.to(device))
    opt = torch.optim.AdamW(head.parameters(), lr=CFG["probe_lr"],
                            weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["probe_epochs"])

    @torch.no_grad()
    def evaluate(X, y, s):
        head.eval()
        logits = head(X)
        return (aggregate_subject(torch.softmax(logits, 1).cpu().numpy(), s, y.numpy()),
                crit(logits, y.to(device)).item())

    best, best_state, best_epoch, wait, peek = -1.0, None, 0, 0, -1.0
    for e in range(CFG["probe_epochs"]):
        head.train(); tot = 0.0
        for xb, yb in dl:
            opt.zero_grad(); loss = crit(head(xb), yb)
            loss.backward(); opt.step(); tot += loss.item() * len(xb)
        sched.step()
        m_es, es_loss = evaluate(es_X, es_y, es_s)
        if CFG["report_peek_bias"] and CFG["nested_early_stop"]:
            peek = max(peek, evaluate(va_X, va_y, va_s)[0]["macro_f1"])
        if verbose and (e == 0 or e % 5 == 4):
            print(f"    [p {e+1:2d}] train_loss={tot/len(ds):.4f}  inner_loss={es_loss:.4f}  "
                  f"inner_macroF1={m_es['macro_f1']:.4f}")
        if m_es["macro_f1"] > best:
            best, best_epoch, wait = m_es["macro_f1"], e + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
        else:
            wait += 1
            if wait >= CFG["probe_patience"]:
                if verbose: print(f"    early stop @ epoch {e+1}")
                break

    head.load_state_dict(best_state)
    out, _ = evaluate(va_X, va_y, va_s)          # the outer fold, scored once
    out["best_epoch"] = best_epoch
    out["inner_macro_f1"] = best
    out["state_dict"] = best_state
    out["standardiser"] = (mu, sd)
    if CFG["report_peek_bias"]:
        out["peek_macro_f1"] = peek if CFG["nested_early_stop"] else out["macro_f1"]
    return out

F = cached_features(central)
probe_results = {}
for f in CFG["run_folds"]:
    print(f"\n=== probe fold {f} ({int((F['fold_det'] == f).sum()):,} val images) ===")
    probe_results[f] = train_probe(F, f)
    print(f"  stopped @ epoch {probe_results[f]['best_epoch']} "
          f"(inner macroF1={probe_results[f]['inner_macro_f1']:.4f}) "
          f"-> OUTER macroF1={probe_results[f]['macro_f1']:.4f}")

summarise(probe_results, f"PROBE over folds {list(probe_results)}")
if len(CFG["run_folds"]) < CFG["n_folds"]:
    print(f"\n!! {len(CFG['run_folds'])} of {CFG['n_folds']} folds run. One fold holds ~4 "
          f"mild_moderate subjects, so this number is mostly noise.\n"
          f"   Set CFG['run_folds'] = [0,1,2,3,4] -- the probe is cheap enough that there "
          f"is no reason not to.")

# To reuse section 7's per-class report and confusion matrices on these results:
#   results = probe_results


## 5c. Reference probe (scikit-learn) — the number every change must beat

Section 5b trains the head with AdamW over epochs. Logistic regression on the same
cached features solves the same problem in closed form, in seconds, with one
interpretable knob (`C`, inverse L2 strength). That makes it the honest floor: if a
fine-tuned network cannot beat a linear model on frozen ImageNet features, the
fine-tuning is hurting, not helping.

Two things this cell decides for you:

- **Aggregation.** `prob_mean` classifies each slice and averages the softmax per
  subject. `feature_mean` averages the subject's 512-d vectors first and classifies
  once — fewer, cleaner rows, and it measured better here.
- **Whether your augmentation helps.** The second sweep adds the cached augmented
  views to the training rows. A recipe that regularises leaves the score flat or
  raises it; a recipe that destroys signal drops it. This is the check that caught
  `RandomResizedCrop`.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score

def _np_views(F):
    det = (F["X_det"].numpy(), F["y_det"].numpy(), np.asarray(F["s_det"]), F["fold_det"].numpy())
    aug = None
    if len(F["X_aug"]):
        aug = (F["X_aug"].numpy(), F["y_aug"].numpy(), np.asarray(F["s_aug"]), F["fold_aug"].numpy())
    return det, aug

def group_mean(X, y, keys):
    """Collapse rows sharing a key (e.g. a subject) to their mean vector."""
    u, inv = np.unique(keys, return_inverse=True)
    M = np.zeros((len(u), X.shape[1]))
    np.add.at(M, inv, X)
    M /= np.bincount(inv)[:, None]
    lab = np.zeros(len(u), dtype=y.dtype); lab[inv] = y
    return M.astype(np.float32), lab, u

def _fit_lr(X, y, C):
    sc = StandardScaler().fit(X)
    return sc, LogisticRegression(C=C, max_iter=3000, class_weight="balanced").fit(sc.transform(X), y)

def subject_pool(prob, subs, y, classes):
    """Mean the per-slice probabilities over each subject -> one prediction per person."""
    u, inv = np.unique(subs, return_inverse=True)
    P = np.zeros((len(u), prob.shape[1])); np.add.at(P, inv, prob); P /= np.bincount(inv)[:, None]
    true = np.zeros(len(u), dtype=y.dtype); true[inv] = y
    return true, classes[P.argmax(1)], P

def score3(true, pred):
    return dict(macro_f1=f1_score(true, pred, average="macro"),
                bal_acc=balanced_accuracy_score(true, pred),
                qwk=cohen_kappa_score(true, pred, weights="quadratic"))

def sk_probe(F, C, agg="prob_mean", use_aug=False, folds=None):
    """Frozen-feature logistic regression scored at subject level. Seconds per fold.

    agg="prob_mean"     classify every slice, then average the softmax per subject.
    agg="feature_mean"  average a subject's feature vectors first, classify once.
    use_aug=True        add the cached augmented views to the training rows.

    `use_aug` is the augmentation test: a recipe that regularises should leave the
    score roughly flat or raise it. On the old RandomResizedCrop recipe, training on
    augmented views alone scored 0.35 against 0.58 deterministic -- the crop was
    randomising brain-to-skull scale, which IS the atrophy signal. Re-run this after
    any change to build_tf and refuse a recipe that costs you macro-F1.
    """
    folds = list(range(CFG["n_folds"])) if folds is None else folds
    (Xd, yd, sd, fd), aug = _np_views(F)
    out = []
    for k in folds:
        tr, va = fd != k, fd == k
        Xtr, ytr, ktr = Xd[tr], yd[tr], sd[tr]
        if use_aug and aug is not None:
            Xa, ya, sa, fa = aug
            m = fa != k
            view = (np.arange(len(Xa))[m] // len(Xd)).astype(str)   # a view is a distinct row
            Xtr = np.concatenate([Xtr, Xa[m]]); ytr = np.concatenate([ytr, ya[m]])
            ktr = np.concatenate([ktr, np.char.add(sa[m].astype(str), view)])

        if agg == "feature_mean":
            Xtr, ytr, _ = group_mean(Xtr, ytr, ktr)
            Xva, true, _ = group_mean(Xd[va], yd[va], sd[va])
            sc, clf = _fit_lr(Xtr, ytr, C)
            pred = clf.predict(sc.transform(Xva))
        else:
            sc, clf = _fit_lr(Xtr, ytr, C)
            true, pred, _ = subject_pool(clf.predict_proba(sc.transform(Xd[va])),
                                         sd[va], yd[va], clf.classes_)
        out.append((score3(true, pred), true, pred))
    return out

def sk_probe_sweep(F, use_aug=False, folds=None):
    print(f"{'agg':<14}{'C':>8}   macro-F1          bal-acc  QWK")
    best = None
    for agg in ("prob_mean", "feature_mean"):
        for C in CFG["sk_c_grid"]:
            r = [m for m, _, _ in sk_probe(F, C, agg, use_aug, folds)]
            f1 = np.mean([x["macro_f1"] for x in r])
            print(f"{agg:<14}{C:>8.4g}   {f1:.3f} +/- {np.std([x['macro_f1'] for x in r]):.3f}   "
                  f"{np.mean([x['bal_acc'] for x in r]):.3f}    {np.mean([x['qwk'] for x in r]):.3f}")
            if best is None or f1 > best[0]:
                best = (f1, agg, C)
    print(f"\nbest: {best[1]} @ C={best[2]:.4g} -> subject macro-F1 {best[0]:.3f}")
    print("majority baseline: macro-F1 0.289 / bal-acc 0.333")
    print("C is picked on the same folds it is scored on, so read this as optimistic by\n"
          "~0.02-0.04. Freeze (agg, C) here; fold -1 gets touched once, at the very end.")
    return best

def report_folds(rows, title):
    f1 = [m["macro_f1"] for m, _, _ in rows]
    print(f"\n=== {title} ===")
    print(f"subject macro-F1 : {np.mean(f1):.3f} +/- {np.std(f1):.3f}   {[round(v, 3) for v in f1]}")
    print(f"subject bal-acc  : {np.mean([m['bal_acc'] for m, _, _ in rows]):.3f}")
    print(f"quadratic kappa  : {np.mean([m['qwk'] for m, _, _ in rows]):.3f}")
    true = np.concatenate([t for _, t, _ in rows]); pred = np.concatenate([p for _, _, p in rows])
    print(classification_report(true, pred, target_names=CLASS_NAMES, zero_division=0))
    print("pooled confusion (rows=true, cols=pred):"); print(confusion_matrix(true, pred))

def aug_check(F, agg, C, folds=None):
    """Does the current build_tf help or hurt? Same model, same C, one knob changed.

    Only the chosen setting is re-run rather than the whole grid -- this is a yes/no
    question and the augmented matrix is probe_views times larger.
    """
    det = [m["macro_f1"] for m, _, _ in sk_probe(F, C, agg, False, folds)]
    aug = [m["macro_f1"] for m, _, _ in sk_probe(F, C, agg, True, folds)]
    print(f"\naugmentation check ({agg}, C={C:.4g})")
    print(f"  deterministic rows only : {np.mean(det):.3f} +/- {np.std(det):.3f}")
    print(f"  + augmented views       : {np.mean(aug):.3f} +/- {np.std(aug):.3f}")
    drop = np.mean(det) - np.mean(aug)
    verdict = ("HURTING -- a transform in build_tf is destroying signal. Do not train on it."
               if drop > 0.02 else
               "fine -- augmentation is not costing you accuracy.")
    print(f"  delta {drop:+.3f}  ->  {verdict}")
    return np.mean(det), np.mean(aug)

sk_best = sk_probe_sweep(F, use_aug=False)
aug_check(F, sk_best[1], sk_best[2])
report_folds(sk_probe(F, sk_best[2], sk_best[1]),
             f"reference probe ({sk_best[1]}, C={sk_best[2]:.4g})")


## 5d. Hierarchical cascade — split the 3-way into two easier problems

A flat 3-way softmax treats `no_dementia → mild_moderate` as no worse than
`no_dementia → very_mild`, and it has to learn the rare class against 212 healthy
subjects at once. The cascade splits it:

- **stage A** — demented vs not (212 vs 65 subjects): the well-posed, well-populated
  problem, and the one a screening clinic actually asks.
- **stage B** — `very_mild` vs `mild_moderate`, trained only on demented subjects, so
  the healthy majority stops drowning the distinction.

The chain rule recombines them into a genuine 3-class distribution, so you still
report one confusion matrix.

This is a fair fight, not a guaranteed win — two stages mean two chances to be wrong
and the errors compound. The cell prints the cascade next to stage A alone; keep
whichever of this and section 5c wins on the full five folds.


In [ ]:
def sk_cascade(F, C_bin=None, C_fine=None, agg=None, folds=None):
    """Two-stage classifier instead of one flat 3-way softmax.

        stage A: demented vs not          (212 vs 65 dev subjects)
        stage B: very_mild vs mild_moderate, trained on demented subjects only

    The chain rule puts them back together as a real 3-class distribution:

        P(no_dementia)   = P_A(healthy)
        P(very_mild)     = P_A(demented) * P_B(very_mild)
        P(mild_moderate) = P_A(demented) * P_B(mild_moderate)

    Stage B runs on every row at inference and is simply down-weighted by P_A on the
    healthy ones, so no hard threshold is needed and the errors stay differentiable.
    """
    C_bin = CFG["cascade_c_binary"] if C_bin is None else C_bin
    C_fine = CFG["cascade_c_fine"] if C_fine is None else C_fine
    agg = CFG["cascade_agg"] if agg is None else agg
    folds = list(range(CFG["n_folds"])) if folds is None else folds
    (Xd, yd, sd, fd), _ = _np_views(F)
    rows, bin_f1 = [], []
    for k in folds:
        tr, va = fd != k, fd == k
        if agg == "feature_mean":
            Xtr, ytr, _ = group_mean(Xd[tr], yd[tr], sd[tr])
            Xva, true, _ = group_mean(Xd[va], yd[va], sd[va])
        else:
            Xtr, ytr, Xva = Xd[tr], yd[tr], Xd[va]

        scA, A = _fit_lr(Xtr, (ytr > 0).astype(int), C_bin)
        dem = ytr > 0
        scB, B = _fit_lr(Xtr[dem], ytr[dem], C_fine)

        pA = A.predict_proba(scA.transform(Xva))
        pB = B.predict_proba(scB.transform(Xva))
        P = np.zeros((len(Xva), len(CLASS_NAMES)))
        P[:, 0] = pA[:, list(A.classes_).index(0)]
        p_dem = pA[:, list(A.classes_).index(1)]
        for j, c in enumerate(B.classes_):
            P[:, c] = p_dem * pB[:, j]

        if agg == "feature_mean":
            pred = P.argmax(1)
            true_b, pred_b = (true > 0).astype(int), A.classes_[pA.argmax(1)]
        else:
            true, pred, _ = subject_pool(P, sd[va], yd[va], np.arange(len(CLASS_NAMES)))
            true_b, pred_b, _ = subject_pool(pA, sd[va], (yd[va] > 0).astype(int), A.classes_)
        rows.append((score3(true, pred), true, pred))
        bin_f1.append(f1_score(true_b, pred_b, average="macro"))
    return rows, bin_f1

def sk_cascade_sweep(F, agg=None, folds=None):
    """Grid over the two regularisation strengths, at CFG['cascade_agg'] only.

    Sweeping both aggregations would fit 2 x len(grid)^2 x n_folds models and the
    slice-level half is ~50x the rows; set CFG['cascade_agg'] = 'prob_mean' and re-run
    if you want to see that side.
    """
    agg = CFG["cascade_agg"] if agg is None else agg
    print(f"{'agg':<14}{'C_bin':>8}{'C_fine':>9}   macro-F1          bal-acc  QWK")
    best = None
    for Ca in CFG["sk_c_grid"]:
        for Cb in CFG["sk_c_grid"]:
            rows, _ = sk_cascade(F, Ca, Cb, agg, folds)
            f1 = np.mean([m["macro_f1"] for m, _, _ in rows])
            if best is None or f1 > best[0]:
                best = (f1, agg, Ca, Cb)
                print(f"{agg:<14}{Ca:>8.4g}{Cb:>9.4g}   {f1:.3f} +/- "
                      f"{np.std([m['macro_f1'] for m, _, _ in rows]):.3f}   "
                      f"{np.mean([m['bal_acc'] for m, _, _ in rows]):.3f}    "
                      f"{np.mean([m['qwk'] for m, _, _ in rows]):.3f}   <- new best")
    print(f"\nbest: {best[1]} C_bin={best[2]:.4g} C_fine={best[3]:.4g} -> macro-F1 {best[0]:.3f}")
    return best

cas_best = sk_cascade_sweep(F)
rows, bin_f1 = sk_cascade(F, cas_best[2], cas_best[3], cas_best[1])
report_folds(rows, f"hierarchical cascade ({cas_best[1]}, "
                   f"C_bin={cas_best[2]:.4g}, C_fine={cas_best[3]:.4g})")
print(f"\nstage A alone -- demented vs not, the number a clinician actually screens on:")
print(f"  subject macro-F1 {np.mean(bin_f1):.3f} +/- {np.std(bin_f1):.3f}   "
      f"{[round(v, 3) for v in bin_f1]}")
print("\nThe cascade is not automatically better than the flat probe on macro-F1 -- the\n"
      "two stages' errors compound. Compare it against `sk_best` above and keep the\n"
      "winner. It usually wins on QWK (it cannot confuse no_dementia with\n"
      "mild_moderate as easily), and stage A is a strong standalone result either way.")


## 6. Two-stage training over the selected fold(s)

In [ ]:
from torch.utils.data import WeightedRandomSampler

def make_train_loader(tr):
    """Training loader. With use_sampler, draw rare-class images more often
    (WeightedRandomSampler) — the image-friendly way to fight class imbalance,
    instead of inventing fake images (SMOTE/ADASYN don't fit raw images)."""
    ds = OasisDS(tr, True)
    if CFG["use_sampler"]:
        cw = class_weights(tr).cpu().numpy()          # weight per class
        sample_w = cw[tr["label"].to_numpy()]         # weight per image
        sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)
        return DataLoader(ds, CFG["batch_size"], sampler=sampler,
                          num_workers=CFG["num_workers"], pin_memory=True, drop_last=True)
    return DataLoader(ds, CFG["batch_size"], shuffle=True,
                      num_workers=CFG["num_workers"], pin_memory=True, drop_last=True)

def train_fold(fold):
    """Train on the outer fold's training subjects, minus an inner stopping set.

    The outer fold is evaluated exactly once, with the weights the inner set chose.
    Set CFG["nested_early_stop"]=False to reproduce the old (optimistic) behaviour.
    """
    dev = central[(central.fold != fold) & (central.fold != -1)]
    va = central[central.fold == fold]
    if CFG["nested_early_stop"]:
        inner = inner_val_subjects(dev.subject_id, dev.label)
        tr, es = dev[~dev.subject_id.isin(inner)], dev[dev.subject_id.isin(inner)]
    else:
        tr, es = dev, va
    print(f"\n=== fold {fold}: train {tr.subject_id.nunique()} subj / {len(tr)} img  "
          f"| inner-stop {es.subject_id.nunique()} subj  "
          f"| OUTER val {va.subject_id.nunique()} subj / {len(va)} img ===")

    tr_dl = make_train_loader(tr)
    mk = lambda d: DataLoader(OasisDS(d, False), CFG["batch_size"], shuffle=False,
                              num_workers=CFG["num_workers"], pin_memory=True)
    es_dl, va_dl = mk(es), mk(va)

    model = build_model(CFG["backbone"]).to(device)
    # If the sampler already balances classes, don't also weight the loss (would double-count).
    criterion = nn.CrossEntropyLoss(
        label_smoothing=CFG["label_smoothing"],
        weight=None if CFG["use_sampler"] else class_weights(tr))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    peek = -1.0

    def checkpoint():
        """Score the inner set; track (but never act on) the outer fold."""
        nonlocal peek
        m = subject_eval(model, es_dl, criterion)
        if CFG["report_peek_bias"] and CFG["nested_early_stop"]:
            peek = max(peek, subject_eval(model, va_dl)["macro_f1"])
        return m

    # Stage 1 -- frozen backbone, head only.
    set_trainable(model, CFG["backbone"], "head")
    opt = torch.optim.AdamW(param_groups(model, CFG["backbone"], CFG["lr_backbone"], CFG["lr_head"]),
                            weight_decay=CFG["weight_decay"])
    for e in range(CFG["stage1_epochs"]):
        l = run_epoch(model, tr_dl, criterion, opt, scaler)
        print(f"  [s1 {e+1}] train_loss={l:.4f}")

    # Baseline: the stage-1 head is often already competitive. Seed `best` with it so
    # stage 2 can only ever be kept if it genuinely beats the frozen-backbone model.
    m0 = checkpoint()
    print(f"  [s1 end] inner_macroF1={m0['macro_f1']:.4f}  inner_balAcc={m0['bal_acc']:.4f}")
    best = m0["macro_f1"]
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_tag, wait = "stage1", 0

    # Stage 2 -- unfreeze from CFG["freeze_until"] onward, discriminative LRs, cosine decay.
    set_trainable(model, CFG["backbone"], CFG["freeze_until"])
    opt = torch.optim.AdamW(param_groups(model, CFG["backbone"], CFG["lr_backbone"], CFG["lr_head_ft"]),
                            weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["stage2_epochs"])
    for e in range(CFG["stage2_epochs"]):
        l = run_epoch(model, tr_dl, criterion, opt, scaler)
        m = checkpoint()
        sched.step()
        gap = m["val_loss"] - l          # widening gap = the net is memorising the train set
        print(f"  [s2 {e+1}] train_loss={l:.4f}  inner_loss={m['val_loss']:.4f}  gap={gap:+.3f}  "
              f"inner_macroF1={m['macro_f1']:.4f}  inner_balAcc={m['bal_acc']:.4f}")
        if m["macro_f1"] > best:
            best = m["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_tag, wait = f"s2e{e+1}", 0
        else:
            wait += 1
            if wait >= CFG["patience"]: print("  early stop"); break

    model.load_state_dict(best_state)
    out = subject_eval(model, va_dl, criterion)      # the outer fold, scored once
    out["inner_macro_f1"] = best
    if CFG["report_peek_bias"]:
        out["peek_macro_f1"] = peek if CFG["nested_early_stop"] else out["macro_f1"]
    ckpt = f"/kaggle/working/model_fold{fold}.pt"
    torch.save({"state_dict": best_state, "cfg": CFG, "fold": fold,
                "inner_macro_f1": best, "outer_macro_f1": out["macro_f1"],
                "best_epoch": best_tag, "class_names": CLASS_NAMES}, ckpt)
    print(f"  stopped @ {best_tag} (inner macroF1={best:.4f}) "
          f"-> OUTER macroF1={out['macro_f1']:.4f}   saved -> {ckpt}")
    return model, out

results = {}
for f in CFG["run_folds"]:
    _, results[f] = train_fold(f)

## 7. Results (subject-level, vs 76.7% majority baseline)

The headline row is the **outer** fold, scored once by weights an inner split chose.
When `report_peek_bias` is on, the old best-on-the-reported-fold number is printed
underneath it — that gap is how much of the previous headline was selection, not
signal. Report the honest row; cite the gap in the paper as a methods contribution.


In [ ]:
summarise(results, "SECTION 6 -- fine-tuned model, subject level")

for f in results:
    print(f"--- fold {f} ---"); print(results[f]["report"])
    print("confusion matrix (rows=true, cols=pred):"); print(results[f]["cm"], "\n")


## 8. Persist outputs (so a dead session can't eat your checkpoints)

`/kaggle/working` is scratch on the running container. It is wiped when the session
ends **and when you change the accelerator** — that is what deleted `model_fold0.pt`.

Two things that always work, in priority order:

1. **Download the zip the cell below prints.** One click, no setup, no internet
   needed, nothing to forget. Do this every time a run finishes.
2. **Save Version → "Save & Run All (Commit)".** Kaggle re-runs the notebook
   top-to-bottom on a clean machine and whatever lands in `/kaggle/working`
   becomes a permanent Output — downloadable later, and attachable as an input
   dataset to another notebook. This is the closest thing to "automatic": you
   press it once and it persists the run without further babysitting.

A third option, **Settings → Persistence → "Files only"**, keeps `/kaggle/working`
across restarts of this notebook. If you see it in your sidebar, turn it on — but
it is not present in every Kaggle UI/notebook type, and Kaggle treats it as
best-effort (it can fail on crashes or large files). Treat it as a bonus, never
as your only copy.

**Do not rely on the accelerator dropdown being safe.** Set it once, before you
start, and leave it alone for the rest of the session.


In [ ]:
import json, zipfile
from pathlib import Path
from IPython.display import FileLink, display

WORK      = Path("/kaggle/working")
ARTIFACTS = ("*.pt", "*.parquet", "*.csv")      # what's worth keeping (outputs.zip excluded)


def persist_outputs(extra_meta=None):
    """Bundle every artifact in /kaggle/working and hand back a download link."""
    files = sorted({p for pat in ARTIFACTS for p in WORK.glob(pat)})
    if not files:
        print("nothing to persist -- /kaggle/working has no artifacts yet")
        return None

    print("artifacts in /kaggle/working:")
    for f in files:
        print(f"  {f.name:44s} {f.stat().st_size/1e6:8.2f} MB")

    # A small run-summary rides along so a downloaded zip is self-describing.
    meta = WORK / "run_summary.json"
    meta.write_text(json.dumps({
        "cfg": {k: str(v) for k, v in CFG.items()},
        "folds": list(results),
        "macro_f1": {str(k): float(results[k]["macro_f1"]) for k in results},
        "bal_acc":  {str(k): float(results[k]["bal_acc"])  for k in results},
        **(extra_meta or {}),
    }, indent=2))

    bundle = WORK / "outputs.zip"
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files + [meta]:
            z.write(f, f.name)
    print(f"\nbundled -> {bundle.name} ({bundle.stat().st_size/1e6:.2f} MB)")

    print("\n" + "=" * 72)
    print(">>> CLICK THE LINK BELOW NOW. /kaggle/working is erased when this")
    print(">>> session ends or the accelerator changes -- there is no undo.")
    print("=" * 72)
    display(FileLink(bundle.name))          # FileLink paths are relative to /kaggle/working
    print("\nThen, to keep a permanent copy on Kaggle itself:")
    print("  Save Version -> 'Save & Run All (Commit)'  -> becomes a downloadable Output.")
    return bundle


persist_outputs()

## 9. Diagnostics

In [ ]:
# =============================================================================
# SECTION 9 - ONE-SHOT DIAGNOSTIC BATTERY
# Paste as ONE new cell at the end of train_kaggle.ipynb and run it.
# Requires cells 0-16 to have already run (CFG, central, F, build_model, extract,
# feature_extractor, OasisDS, set_trainable, param_groups, freeze_frozen_bn,
# class_weights, series_id, set_seed, device, AMP, CLASS_NAMES all in scope).
# Everything is printed AND written to /kaggle/working/diagnostics.log
# =============================================================================
import sys, time, copy, traceback, json
import numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (f1_score, balanced_accuracy_score, roc_auc_score,
                             confusion_matrix, classification_report)

RUN = dict(audit=True, geometry=True, leakage=True, permutation=True,
           probe_variants=True, aggregation=True, subtasks=True,
           slice_window=True, train_ceiling=True, backbones=True, finetune=True)

FT_FOLD, FT_S1, FT_S2, FT_PATIENCE = 0, 2, 8, 4
ALT_BACKBONES, ALT_VIEWS = ["resnet50", "efficientnet_b0"], 2
PROBE_EPOCHS, PROBE_PATIENCE, FIXED_EPOCH = 30, 8, 10
ALL_FOLDS = list(range(CFG["n_folds"]))
NV = CFG["probe_views"]

LOGPATH = Path("/kaggle/working/diagnostics.log")
_real_stdout = sys.stdout
class _Tee:
    def __init__(self, p, o): self.f = open(p, "w", encoding="utf-8"); self.o = o
    def write(self, s): self.f.write(s); self.o.write(s)
    def flush(self): self.f.flush(); self.o.flush()

T0 = time.time()
ROWS = []

def banner(t):
    print(f"\n{'='*78}\n### {t}    [t+{time.time()-T0:.0f}s]\n{'='*78}")

def block(name, fn):
    if not RUN.get(name, False):
        print(f"\n--- SKIPPED block: {name}"); return
    t = time.time()
    try:
        fn()
    except Exception:
        print(f"\n!!! BLOCK {name} FAILED - continuing\n{traceback.format_exc()}")
    print(f"--- block {name} took {time.time()-t:.0f}s")

def logrow(**kw): ROWS.append(kw)

def tsel(X, m):
    """Boolean-select rows of a torch tensor or numpy array with a bool mask."""
    m = np.asarray(m)
    return X[torch.as_tensor(m)] if torch.is_tensor(X) else np.asarray(X)[m]

# ---------------------------------------------------------------- scoring ---
def ovr_auc(P, t, nc):
    out = []
    for c in range(nc):
        yb = (t == c).astype(int)
        out.append(roc_auc_score(yb, P[:, c]) if 0 < yb.sum() < len(yb) else np.nan)
    return np.array(out)

def score(P, t, nc=3):
    L = list(range(nc)); pred = P.argmax(1)
    return dict(macro_f1=f1_score(t, pred, average="macro", labels=L, zero_division=0),
                bal_acc=balanced_accuracy_score(t, pred),
                acc=float((pred == t).mean()),
                f1c=f1_score(t, pred, average=None, labels=L, zero_division=0),
                auc=ovr_auc(P, t, nc),
                cm=confusion_matrix(t, pred, labels=L), pred=pred, true=t)

def subj_pack(p, s, t, nc=3, how="mean"):
    cols = [f"c{i}" for i in range(nc)]
    d = pd.DataFrame(p, columns=cols); d["s"] = np.asarray(s); d["y"] = t
    g = d.groupby("s", sort=True)
    if how == "mean":
        P = g[cols].mean().values
    elif how == "median":
        P = g[cols].median().values
    elif how == "geom":
        dl = pd.DataFrame(np.log(np.clip(p, 1e-9, 1.0)), columns=cols)
        dl["s"] = np.asarray(s)
        P = np.exp(dl.groupby("s", sort=True)[cols].mean().values)
        P = P / P.sum(1, keepdims=True)
    elif how == "vote":
        d2 = d.copy(); d2["a"] = p.argmax(1); g2 = d2.groupby("s", sort=True)["a"]
        P = np.stack([g2.apply(lambda v, c=c: float((v == c).mean())).values
                      for c in range(nc)], 1)
    else:
        raise ValueError(how)
    return P, g["y"].first().values, np.array(sorted(set(np.asarray(s))), dtype=object)

def fmt(m):
    return (f"F1={m['macro_f1']:.4f} bal={m['bal_acc']:.4f} acc={m['acc']:.4f} "
            f"f1c=[{','.join(f'{v:.2f}' for v in m['f1c'])}] "
            f"auc=[{','.join('nan' if np.isnan(v) else f'{v:.3f}' for v in m['auc'])}]")

# ------------------------------------------------------------- probe core ---
def fit_probe(tr_X, tr_y, va_X, va_y, va_s, *, nc=3, head="linear", sampler=True,
              class_weight=False, epochs=PROBE_EPOCHS, patience=PROBE_PATIENCE,
              lr=CFG["probe_lr"], seed=0, extra=None, verbose=False):
    """extra=(X,y,s) is scored at the best epoch (used for the train-set ceiling)."""
    torch.manual_seed(seed)
    mu, sd = tr_X.mean(0, keepdim=True), tr_X.std(0, keepdim=True).clamp_min(1e-6)
    trX = ((tr_X - mu) / sd).to(device); vaX = ((va_X - mu) / sd).to(device)
    trY = tr_y.to(device)
    counts = torch.bincount(tr_y, minlength=nc).float()
    cw = counts.sum() / (nc * counts.clamp_min(1.0))
    ds = TensorDataset(trX, trY)
    dl = (DataLoader(ds, 256, sampler=WeightedRandomSampler(cw[tr_y].double(),
                                                            len(tr_y), True))
          if sampler else DataLoader(ds, 256, shuffle=True))
    D = trX.shape[1]
    net = (nn.Sequential(nn.Dropout(CFG["dropout"]), nn.Linear(D, nc)) if head == "linear"
           else nn.Sequential(nn.Dropout(CFG["dropout"]), nn.Linear(D, 256), nn.ReLU(),
                              nn.Dropout(CFG["dropout"]), nn.Linear(256, nc))).to(device)
    crit = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"],
                               weight=cw.to(device) if class_weight else None)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=CFG["weight_decay"])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    hist, best, best_e, wait, best_state = [], -1.0, 1, 0, None
    vy = np.asarray(va_y.numpy() if torch.is_tensor(va_y) else va_y)
    for e in range(epochs):
        net.train(); tot = 0.0
        for xb, yb in dl:
            opt.zero_grad(); l = crit(net(xb), yb); l.backward(); opt.step()
            tot += l.item() * len(xb)
        sch.step(); net.eval()
        with torch.no_grad():
            logits = net(vaX).float().cpu()
        p = torch.softmax(logits, 1).numpy()
        P, ts, _ = subj_pack(p, va_s, vy, nc)
        mi, ms = score(p, vy, nc), score(P, ts, nc)
        hist.append(dict(epoch=e + 1, train_loss=tot / len(ds), img=mi, subj=ms,
                         logits=logits))
        if verbose:
            print(f"      [{e+1:2d}] tr={tot/len(ds):.4f} imgF1={mi['macro_f1']:.4f} "
                  f"subjF1={ms['macro_f1']:.4f}")
        if ms["macro_f1"] > best:
            best, best_e, wait = ms["macro_f1"], e + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break

    out = dict(n_epochs=len(hist), best_epoch=best_e, best=hist[best_e - 1],
               final=hist[-1], fixed=hist[min(FIXED_EPOCH, len(hist)) - 1],
               state=best_state, mu=mu, sd=sd, nc=nc, va_s=np.asarray(va_s), va_y=vy)
    if extra is not None:
        net.load_state_dict(best_state); net.eval()
        eX, ey, es = extra
        eyn = ey.numpy() if torch.is_tensor(ey) else np.asarray(ey)
        with torch.no_grad():
            ep = torch.softmax(net(((eX - mu) / sd).to(device)).float().cpu(), 1).numpy()
        EP, et, _ = subj_pack(ep, es, eyn, nc)
        out["extra_img"], out["extra_subj"] = score(ep, eyn, nc), score(EP, et, nc)
    del trX, vaX; torch.cuda.empty_cache()
    return out

def fold_tensors(Fd, fold, use_aug=True):
    tr_d = (Fd["fold_det"] != fold).numpy(); va_d = (Fd["fold_det"] == fold).numpy()
    has_aug = use_aug and len(Fd["X_aug"]) > 0
    if has_aug:
        tr_a = (Fd["fold_aug"] != fold).numpy()
        trX = torch.cat([tsel(Fd["X_det"], tr_d), tsel(Fd["X_aug"], tr_a)])
        trY = torch.cat([tsel(Fd["y_det"], tr_d), tsel(Fd["y_aug"], tr_a)])
    else:
        trX, trY = tsel(Fd["X_det"], tr_d), tsel(Fd["y_det"], tr_d)
    return (trX, trY, tsel(Fd["X_det"], va_d), tsel(Fd["y_det"], va_d),
            tsel(Fd["s_det"], va_d), tr_d, va_d)

def sweep(tag, Fd, folds=ALL_FOLDS, use_aug=True, **kw):
    accB, accF, accX, aucs = [], [], [], []
    for f in folds:
        trX, trY, vaX, vaY, vaS, _, _ = fold_tensors(Fd, f, use_aug)
        r = fit_probe(trX, trY, vaX, vaY, vaS, **kw)
        b, fi, fx = r["best"]["subj"], r["final"]["subj"], r["fixed"]["subj"]
        print(f"  [{tag}] fold{f} ep={r['n_epochs']:2d} best@{r['best_epoch']:2d} | "
              f"BEST {fmt(b)} | FINAL F1={fi['macro_f1']:.4f} | "
              f"FIX{FIXED_EPOCH} F1={fx['macro_f1']:.4f} | "
              f"img F1={r['best']['img']['macro_f1']:.4f} "
              f"auc=[{','.join('nan' if np.isnan(v) else f'{v:.3f}' for v in r['best']['img']['auc'])}]")
        accB.append(b["macro_f1"]); accF.append(fi["macro_f1"]); accX.append(fx["macro_f1"])
        aucs.append(b["auc"])
    print(f"  [{tag}] MEAN subjF1 best={np.mean(accB):.4f}+/-{np.std(accB):.4f}  "
          f"final={np.mean(accF):.4f}  fix{FIXED_EPOCH}={np.mean(accX):.4f}  "
          f"subjAUC={np.round(np.nanmean(np.stack(aucs), 0), 3).tolist()}")
    logrow(test=tag, best=round(float(np.mean(accB)), 4), sd=round(float(np.std(accB)), 4),
           final=round(float(np.mean(accF)), 4), fixed=round(float(np.mean(accX)), 4),
           auc=np.round(np.nanmean(np.stack(aucs), 0), 3).tolist())
    return accB

# =============================================================== BLOCK 0 ====
def b_audit():
    banner("0. ENVIRONMENT + SPLIT AUDIT")
    import sklearn, torchvision
    print("torch", torch.__version__, "| sklearn", sklearn.__version__,
          "| torchvision", torchvision.__version__, "| cuda", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
    print("\nCFG:", json.dumps({k: str(v) for k, v in CFG.items()}, indent=1))

    dev = central[central.fold != -1].reset_index(drop=True)
    tst = central[central.fold == -1]
    print(f"\ncentral rows={len(central):,} subjects={central.subject_id.nunique()}")
    print(f"dev rows={len(dev):,} subj={dev.subject_id.nunique()} | "
          f"test rows={len(tst):,} subj={tst.subject_id.nunique()}")
    print("dev/test subject overlap:", len(set(dev.subject_id) & set(tst.subject_id)),
          "(MUST be 0)")
    print("subjects in >1 fold  :", int((central.groupby("subject_id")["fold"]
                                         .nunique() > 1).sum()), "(MUST be 0)")
    print("subjects with >1 label:", int((central.groupby("subject_id")["label"]
                                          .nunique() > 1).sum()), "(MUST be 0)")
    print("duplicate filepaths  :", int(central.filepath.duplicated().sum()), "(MUST be 0)")

    sub = central.groupby("subject_id").agg(label=("label", "first"), fold=("fold", "first"))
    ren = {i: n for i, n in enumerate(CLASS_NAMES)}
    print("\nSUBJECT counts per fold x class  <-- the real sample size:")
    print(pd.crosstab(sub.fold, sub.label).rename(columns=ren).to_string())
    print("\nIMAGE counts per fold x class:")
    print(pd.crosstab(central.fold, central.label).rename(columns=ren).to_string())

    ips = central.groupby("subject_id").size()
    print(f"\nimages/subject: min={ips.min()} med={ips.median():.0f} "
          f"mean={ips.mean():.1f} max={ips.max()}")
    ser = pd.Series(series_id(central.filepath.tolist()), index=central.index)
    nser = central.assign(_s=ser).groupby("subject_id")["_s"].nunique()
    print(f"scan-series/subject: min={nser.min()} med={nser.median():.0f} "
          f"mean={nser.mean():.2f} max={nser.max()} | total series={ser.nunique():,}")
    print("\nslice_idx counts:")
    print(central.slice_idx.value_counts().sort_index().to_string())

    print("\nFEATURE-CACHE ALIGNMENT (all MUST be True):")
    print("  len match     :", len(F["X_det"]) == len(dev))
    print("  labels match  :", bool((F["y_det"].numpy() == dev.label.values).all()))
    print("  subjects match:", bool((np.asarray(F["s_det"]) == dev.subject_id.values).all()))
    print("  folds match   :", bool((F["fold_det"].numpy() == dev.fold.values).all()))
    print("  X_det", tuple(F["X_det"].shape), "X_aug", tuple(F["X_aug"].shape),
          "| NaN:", bool(torch.isnan(F["X_det"]).any()),
          f"| norm mean={F['X_det'].norm(dim=1).mean():.3f} "
          f"std={F['X_det'].norm(dim=1).std():.3f}")

# =============================================================== BLOCK 1 ====
def b_geometry():
    banner("1. FEATURE GEOMETRY - how redundant are slices of the same subject?")
    dev = central[central.fold != -1].reset_index(drop=True)
    X = torch.nn.functional.normalize(F["X_det"].float(), dim=1).to(device)
    subj = dev.subject_id.values
    ser = np.array(series_id(dev.filepath.tolist()))
    lab = dev.label.values
    n = len(X); B = 512
    same_s, same_r, same_l, si, so = [], [], [], [], []
    for i in range(0, n, B):
        S = (X[i:i + B] @ X.T).float()
        rows = torch.arange(S.shape[0], device=device)
        j = np.arange(i, min(i + B, n))
        S[rows, torch.as_tensor(j, device=device)] = -2.0
        k = S.argmax(1).cpu().numpy()
        same_s.append(subj[k] == subj[j]); same_r.append(ser[k] == ser[j])
        same_l.append(lab[k] == lab[j])
        msame = torch.as_tensor(subj[None, :] == subj[j][:, None], device=device)
        S[rows, torch.as_tensor(j, device=device)] = float("nan")
        NANS = torch.full_like(S, float("nan"))
        si.append(torch.nanmean(torch.where(msame, S, NANS), 1).cpu())
        so.append(torch.nanmean(torch.where(~msame, S, NANS), 1).cpu())
    same_s, same_r, same_l = map(np.concatenate, (same_s, same_r, same_l))
    si, so = torch.cat(si).numpy(), torch.cat(so).numpy()
    print(f"1-NN over all {n:,} dev images (self excluded):")
    print(f"  nearest neighbour is same SUBJECT     : {same_s.mean():.4f}")
    print(f"  nearest neighbour is same SCAN SERIES : {same_r.mean():.4f}")
    print(f"  nearest neighbour is same LABEL       : {same_l.mean():.4f}")
    print(f"mean cosine: within-subject={np.nanmean(si):.4f} "
          f"between-subject={np.nanmean(so):.4f} gap={np.nanmean(si)-np.nanmean(so):.4f}")
    logrow(test="1nn_same_subject", best=round(float(same_s.mean()), 4))
    logrow(test="1nn_same_label", best=round(float(same_l.mean()), 4))
    del X, S; torch.cuda.empty_cache()

# =============================================================== BLOCK 2 ====
def b_leakage():
    banner("2. LEAKAGE TEST - subject-grouped split vs random image-level split")
    print("Same features, same head, same hyperparameters. ONLY the split changes.")
    n = len(F["X_det"]); rng = np.random.RandomState(0)
    for fold in ALL_FOLDS:
        n_val = int((F["fold_det"] == fold).sum())
        trX, trY, vaX, vaY, vaS, _, _ = fold_tensors(F, fold)
        ra = fit_probe(trX, trY, vaX, vaY, vaS)
        vm = np.zeros(n, bool); vm[rng.permutation(n)[:n_val]] = True; tm = ~vm
        if NV:
            am = np.tile(tm, NV)
            trX2 = torch.cat([tsel(F["X_det"], tm), tsel(F["X_aug"], am)])
            trY2 = torch.cat([tsel(F["y_det"], tm), tsel(F["y_aug"], am)])
        else:
            trX2, trY2 = tsel(F["X_det"], tm), tsel(F["y_det"], tm)
        rb = fit_probe(trX2, trY2, tsel(F["X_det"], vm), tsel(F["y_det"], vm),
                       tsel(F["s_det"], vm))
        print(f"  fold{fold} n_val_img={n_val:,}")
        print(f"    SUBJECT-GROUPED image-level : {fmt(ra['best']['img'])}")
        print(f"    SUBJECT-GROUPED subject-lvl : {fmt(ra['best']['subj'])}")
        print(f"    RANDOM-IMAGE    image-level : {fmt(rb['best']['img'])}   "
              f"<-- what leaky notebooks report")
        logrow(test=f"leak_grouped_img_f{fold}",
               best=round(ra['best']['img']['macro_f1'], 4))
        logrow(test=f"leak_random_img_f{fold}",
               best=round(rb['best']['img']['macro_f1'], 4))

# =============================================================== BLOCK 3 ====
def b_permutation():
    banner("3. LABEL-PERMUTATION TEST - must collapse to chance (macro-F1 ~0.29)")
    dev = central[central.fold != -1].reset_index(drop=True)
    sub = dev.groupby("subject_id")["label"].first()
    shuf = pd.Series(np.random.RandomState(7).permutation(sub.values), index=sub.index)
    ynew = torch.as_tensor(dev.subject_id.map(shuf).values, dtype=torch.long)
    Fp = dict(F); Fp["y_det"] = ynew
    Fp["y_aug"] = ynew.repeat(NV) if NV else F["y_aug"]
    sweep("permuted_labels", Fp)
    print("  If this is well above ~0.29, the split/eval pipeline itself leaks.")

# =============================================================== BLOCK 4 ====
def b_probe_variants():
    banner("4. PROBE VARIANTS on frozen resnet18 features (all 5 folds each)")
    print("BEST=max over epochs (optimistic, current reporting). FINAL=last epoch. "
          f"FIX{FIXED_EPOCH}=epoch {FIXED_EPOCH}. auc=one-vs-rest, threshold-free.\n")
    sweep("v1_linear_sampler",   F, head="linear", sampler=True,  class_weight=False)
    sweep("v2_linear_clsweight", F, head="linear", sampler=False, class_weight=True)
    sweep("v3_linear_plain",     F, head="linear", sampler=False, class_weight=False)
    sweep("v4_mlp_sampler",      F, head="mlp",    sampler=True,  class_weight=False)
    sweep("v5_linear_noaugviews", F, use_aug=False, head="linear", sampler=True)

    print("\n  v6_subject_mean_features (mean of all slice features -> 1 vector/subject):")
    dev = central[central.fold != -1].reset_index(drop=True)
    dfX = pd.DataFrame(F["X_det"].numpy()); dfX["s"] = dev.subject_id.values
    SX = torch.tensor(dfX.groupby("s", sort=True).mean().values, dtype=torch.float32)
    meta = dev.groupby("subject_id", sort=True).agg(label=("label", "first"),
                                                    fold=("fold", "first"))
    SY = torch.tensor(meta.label.values, dtype=torch.long)
    SF, SS = meta.fold.values, meta.index.values
    accs = []
    for f in ALL_FOLDS:
        tr, va = SF != f, SF == f
        r = fit_probe(tsel(SX, tr), tsel(SY, tr), tsel(SX, va), tsel(SY, va),
                      SS[va], epochs=200, patience=40)
        print(f"    fold{f} n_tr={tr.sum()} n_va={va.sum()} BEST {fmt(r['best']['img'])}")
        accs.append(r["best"]["img"]["macro_f1"])
    print(f"    MEAN F1={np.mean(accs):.4f}+/-{np.std(accs):.4f}")
    logrow(test="v6_subject_mean_features", best=round(float(np.mean(accs)), 4))

# =============================================================== BLOCK 5 ====
def b_aggregation():
    banner("5. AGGREGATION + PRIOR CORRECTION + PER-SUBJECT DUMP (v1 probe)")
    dump = []
    for f in ALL_FOLDS:
        trX, trY, vaX, vaY, vaS, _, _ = fold_tensors(F, f)
        r = fit_probe(trX, trY, vaX, vaY, vaS)
        logits = r["best"]["logits"]; p = torch.softmax(logits, 1).numpy()
        vy = vaY.numpy()
        print(f"\n  fold{f} (best epoch {r['best_epoch']} of {r['n_epochs']})")
        for how in ("mean", "median", "geom", "vote"):
            P, t, _ = subj_pack(p, vaS, vy, 3, how)
            print(f"    agg={how:7s} {fmt(score(P, t))}")
        P, t, sids = subj_pack(p, vaS, vy, 3, "mean")
        m = score(P, t)
        print(f"    subject confusion (rows=true, cols=pred):\n"
              f"{np.array2string(m['cm'])}")
        print(f"    subject-level OvR AUC = {np.round(m['auc'], 3).tolist()}   "
              f"<-- KEY: high AUC + low F1 means threshold/prior problem, not signal")
        tp = np.bincount(trY.numpy(), minlength=3) / len(trY)
        vp = np.bincount(vy, minlength=3) / len(vy)
        print(f"    train prior={np.round(tp,3).tolist()} val prior={np.round(vp,3).tolist()}")
        L = logits.numpy()
        for nm, pr in (("TRAIN-prior shift (honest)     ", tp),
                       ("VAL-prior shift (ORACLE bound) ", vp)):
            line = []
            for tau in (0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0):
                La = L + tau * np.log(np.clip(pr, 1e-9, 1))
                pa = np.exp(La - La.max(1, keepdims=True)); pa /= pa.sum(1, keepdims=True)
                Pa, ta, _ = subj_pack(pa, vaS, vy, 3, "mean")
                line.append(f"t={tau}:{score(Pa, ta)['macro_f1']:.3f}")
            print(f"    {nm} " + " ".join(line))
        for sid, tt, pp in zip(sids, t, P):
            dump.append(dict(fold=f, subject=sid, true=CLASS_NAMES[tt],
                             pred=CLASS_NAMES[int(pp.argmax())],
                             p_nd=round(float(pp[0]), 3), p_vm=round(float(pp[1]), 3),
                             p_mm=round(float(pp[2]), 3)))
    D = pd.DataFrame(dump)
    D.to_csv("/kaggle/working/probe_subject_predictions.csv", index=False)
    print(f"\n  per-subject dump -> /kaggle/working/probe_subject_predictions.csv "
          f"({len(D)} rows)")
    print("\n  MISCLASSIFIED SUBJECTS (all folds):")
    print(D[D.true != D.pred].to_string(index=False))
    print("\n  overall subject confusion across folds:")
    print(pd.crosstab(D.true, D.pred).to_string())

# =============================================================== BLOCK 6 ====
def b_subtasks():
    banner("6. SUB-TASK DECOMPOSITION - where does the 3-class confusion live?")
    print("  (a) BINARY no_dementia vs any_dementia   (chance macro-F1 ~0.43)")
    Fb = dict(F); Fb["y_det"] = (F["y_det"] > 0).long()
    Fb["y_aug"] = (F["y_aug"] > 0).long() if len(F["X_aug"]) else F["y_aug"]
    a = []
    for f in ALL_FOLDS:
        trX, trY, vaX, vaY, vaS, _, _ = fold_tensors(Fb, f)
        r = fit_probe(trX, trY, vaX, vaY, vaS, nc=2)
        print(f"    fold{f} BEST subj {fmt(r['best']['subj'])} | "
              f"img {fmt(r['best']['img'])}")
        a.append(r["best"]["subj"]["macro_f1"])
    print(f"    MEAN subjF1={np.mean(a):.4f}+/-{np.std(a):.4f}")
    logrow(test="binary_nd_vs_rest", best=round(float(np.mean(a)), 4))

    print("\n  (b) BINARY very_mild vs mild_moderate, dementia subjects only "
          "(chance macro-F1 ~0.40)")
    a = []
    fd = F["fold_det"].numpy(); fa = F["fold_aug"].numpy()
    dm = (F["y_det"] > 0).numpy(); am = (F["y_aug"] > 0).numpy()
    for f in ALL_FOLDS:
        tr, va, ta = dm & (fd != f), dm & (fd == f), am & (fa != f)
        trX = torch.cat([tsel(F["X_det"], tr), tsel(F["X_aug"], ta)])
        trY = torch.cat([tsel(F["y_det"], tr) - 1, tsel(F["y_aug"], ta) - 1])
        r = fit_probe(trX, trY, tsel(F["X_det"], va), tsel(F["y_det"], va) - 1,
                      tsel(F["s_det"], va), nc=2)
        print(f"    fold{f} n_va_img={va.sum()} BEST subj {fmt(r['best']['subj'])}")
        a.append(r["best"]["subj"]["macro_f1"])
    print(f"    MEAN subjF1={np.mean(a):.4f}+/-{np.std(a):.4f}")
    logrow(test="binary_vm_vs_mm", best=round(float(np.mean(a)), 4))

# =============================================================== BLOCK 7 ====
def b_slice_window():
    banner("7. SLICE-WINDOW ABLATION (re-uses the same cached features)")
    sl = central[central.fold != -1].reset_index(drop=True).slice_idx.values
    for lo, hi, name in ((CFG["central_lo"], CFG["central_hi"], "full cached window"),
                         (128, 132, "128-132 (old config)"), (130, 130, "single slice 130"),
                         (124, 127, "lower edge"), (133, 136, "upper edge")):
        keep = (sl >= lo) & (sl <= hi)
        if keep.sum() == 0:
            print(f"  [{lo}-{hi}] no rows, skipped"); continue
        ka = np.tile(keep, NV) if NV else None
        Fs = dict(X_det=tsel(F["X_det"], keep), y_det=tsel(F["y_det"], keep),
                  s_det=tsel(F["s_det"], keep), fold_det=tsel(F["fold_det"], keep),
                  X_aug=tsel(F["X_aug"], ka) if NV else F["X_aug"],
                  y_aug=tsel(F["y_aug"], ka) if NV else F["y_aug"],
                  s_aug=tsel(F["s_aug"], ka) if NV else F["s_aug"],
                  fold_aug=tsel(F["fold_aug"], ka) if NV else F["fold_aug"])
        print(f"  --- {name} [{lo}-{hi}]  ({keep.sum():,} det images)")
        sweep(f"slice_{lo}_{hi}", Fs)

# =============================================================== BLOCK 8 ====
def b_train_ceiling():
    banner("8. TRAIN-SET CEILING - can the probe fit its own training subjects?")
    print("  low TRAIN F1  => features lack the signal (underfit, data/feature problem)")
    print("  high TRAIN F1 + low VAL F1 => generalisation gap (model/sample-size problem)")
    for f in ALL_FOLDS:
        trX, trY, vaX, vaY, vaS, tr_d, _ = fold_tensors(F, f)
        r = fit_probe(trX, trY, vaX, vaY, vaS,
                      extra=(tsel(F["X_det"], tr_d), tsel(F["y_det"], tr_d),
                             tsel(F["s_det"], tr_d)))
        print(f"  fold{f}   VAL subj {fmt(r['best']['subj'])}")
        print(f"         TRAIN subj {fmt(r['extra_subj'])}")
        print(f"         TRAIN img  {fmt(r['extra_img'])}")

# =============================================================== BLOCK 9 ====
def b_backbones():
    banner("9. ALTERNATIVE FROZEN BACKBONES")
    dev = central[central.fold != -1].reset_index(drop=True)
    fold_det = dev["fold"].to_numpy()
    if NV >= ALT_VIEWS:
        k = ALT_VIEWS * len(dev)
        Fc = dict(F); Fc["X_aug"] = F["X_aug"][:k]; Fc["y_aug"] = F["y_aug"][:k]
        Fc["s_aug"] = np.asarray(F["s_aug"])[:k]; Fc["fold_aug"] = F["fold_aug"][:k]
        print(f"  control: resnet18 with only {ALT_VIEWS} aug views (matched budget)")
        sweep(f"bb_resnet18_ctl_v{ALT_VIEWS}", Fc)
    for bk in ALT_BACKBONES:
        t = time.time()
        path = Path(f"/kaggle/working/feats_{bk}_{CFG['central_lo']}-{CFG['central_hi']}"
                    f"_g{CFG['slice_gap']}_v{ALT_VIEWS}_seed{CFG['seed']}.pt")
        if path.exists():
            Fb = torch.load(path, weights_only=False); print(f"  loaded {path.name}")
        else:
            print(f"  extracting {bk}: 1 det + {ALT_VIEWS} aug views over {len(dev):,} imgs")
            m = feature_extractor(bk)
            Xd, yd, sd_ = extract(dev, m, False, 1)
            Xa, ya, sa = extract(dev, m, True, ALT_VIEWS)
            del m; torch.cuda.empty_cache()
            Fb = dict(X_det=Xd, y_det=yd, s_det=sd_, fold_det=torch.as_tensor(fold_det),
                      X_aug=Xa, y_aug=ya, s_aug=sa,
                      fold_aug=torch.as_tensor(np.tile(fold_det, ALT_VIEWS)))
            torch.save(Fb, path)
            print(f"  cached -> {path.name}  ({Xd.shape[1]}-d, {time.time()-t:.0f}s)")
        sweep(f"bb_{bk}", Fb)
        del Fb; torch.cuda.empty_cache()

# ============================================================== BLOCK 10 ====
def all_bn_eval(model):
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm): m.eval()

def bn_snapshot(model):
    return {n: (m.running_mean.detach().clone(), m.running_var.detach().clone())
            for n, m in model.named_modules()
            if isinstance(m, nn.modules.batchnorm._BatchNorm) and m.running_mean is not None}

def bn_drift(model, snap, top=6):
    d = dict(model.named_modules())
    rows = [(n,
             (d[n].running_mean - mu).norm().item() / (mu.norm().item() + 1e-8),
             (d[n].running_var - var).norm().item() / (var.norm().item() + 1e-8))
            for n, (mu, var) in snap.items()]
    rows.sort(key=lambda r: -r[1])
    return rows[:top]

@torch.no_grad()
def eval_loader(model, loader, crit, batchstats=False):
    mdl = copy.deepcopy(model) if batchstats else model
    mdl.eval()
    if batchstats:
        for m in mdl.modules():
            if isinstance(m, nn.modules.batchnorm._BatchNorm): m.train()
    ps, ys, ss, tot = [], [], [], 0.0
    for x, y, s in loader:
        lo = mdl(x.to(device))
        tot += crit(lo, y.to(device)).item() * x.size(0)
        ps.append(torch.softmax(lo.float(), 1).cpu().numpy())
        ys.append(y.numpy()); ss.extend(s)
    p = np.concatenate(ps); t = np.concatenate(ys)
    P, ts, _ = subj_pack(p, ss, t)
    if batchstats: del mdl; torch.cuda.empty_cache()
    return dict(loss=tot / len(loader.dataset), img=score(p, t), subj=score(P, ts))

def run_epoch_diag(model, loader, crit, opt, scaler, bn_eval_all):
    model.train(); freeze_frozen_bn(model)
    if bn_eval_all: all_bn_eval(model)
    tot = 0.0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        with torch.autocast(device_type="cuda", enabled=AMP):
            loss = crit(model(x), y)
        if scaler is not None:
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            loss.backward(); opt.step()
        tot += loss.item() * x.size(0)
    return tot / len(loader.dataset)

def probe_head_state(Fd, fold, nc=3):
    """Fit the cached-feature linear probe on `fold`, then fold its input standardiser
    into the weights so the result drops straight into build_model's raw-feature head.

        probe:  W @ ((x - mu) / sd) + b   ==   (W / sd) @ x + (b - (W / sd) @ mu)

    Dropout is inverted, so its train-time scaling is identical in both and needs no
    correction. Fd must come from the same backbone as CFG['backbone'], otherwise the
    weight copy raises on shape mismatch.
    """
    trX, trY, vaX, vaY, vaS, _, _ = fold_tensors(Fd, fold)
    r = fit_probe(trX, trY, vaX, vaY, vaS, nc=nc)
    W, b = r["state"]["1.weight"], r["state"]["1.bias"]
    Wf = W / r["sd"].squeeze(0)
    return dict(weight=Wf, bias=b - Wf @ r["mu"].squeeze(0)), r

def load_head(model, backbone, hs):
    lin = getattr(model, HEAD_ATTR[backbone])[1]
    with torch.no_grad():
        lin.weight.copy_(hs["weight"].to(lin.weight.device))
        lin.bias.copy_(hs["bias"].to(lin.bias.device))

def b_finetune():
    banner(f"10. FINE-TUNE DIAGNOSTICS (fold {FT_FOLD} only, 4 variants)")
    tr = central[(central.fold != FT_FOLD) & (central.fold != -1)]
    va = central[central.fold == FT_FOLD]
    print(f"train {tr.subject_id.nunique()} subj / {len(tr):,} img | "
          f"val {va.subject_id.nunique()} subj / {len(va):,} img")
    va_dl = DataLoader(OasisDS(va, False), CFG["batch_size"], shuffle=False,
                       num_workers=CFG["num_workers"], pin_memory=True)
    HS, pr = probe_head_state(F, FT_FOLD)
    print(f"warm-start head <- cached probe on fold {FT_FOLD}: best@ep{pr['best_epoch']} "
          f"subj {fmt(pr['best']['subj'])}")
    VARIANTS = [
        dict(tag="A_baseline_repro", bn_eval_all=False, sampler=True,  lr_bb=CFG["lr_backbone"]),
        dict(tag="B_allBN_eval",     bn_eval_all=True,  sampler=True,  lr_bb=CFG["lr_backbone"]),
        dict(tag="C_BN+clsweight",   bn_eval_all=True,  sampler=False, lr_bb=CFG["lr_backbone"]),
        dict(tag="D_BN+clsw+lowLR",  bn_eval_all=True,  sampler=False, lr_bb=2e-6),
    ]
    for V in VARIANTS:
        t = time.time()
        print(f"\n--- {V['tag']}: bn_eval_all={V['bn_eval_all']} sampler={V['sampler']} "
              f"lr_backbone={V['lr_bb']}")
        set_seed(CFG["seed"])
        ds = OasisDS(tr, True)
        if V["sampler"]:
            sw = class_weights(tr).cpu().numpy()[tr["label"].to_numpy()]
            dl = DataLoader(ds, CFG["batch_size"],
                            sampler=WeightedRandomSampler(sw, len(sw), True),
                            num_workers=CFG["num_workers"], pin_memory=True, drop_last=True)
        else:
            dl = DataLoader(ds, CFG["batch_size"], shuffle=True,
                            num_workers=CFG["num_workers"], pin_memory=True, drop_last=True)
        model = build_model(CFG["backbone"]).to(device)
        load_head(model, CFG["backbone"], HS)
        crit = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"],
                                   weight=None if V["sampler"] else class_weights(tr))
        scaler = torch.amp.GradScaler("cuda", enabled=AMP)
        mw = eval_loader(model, va_dl, crit)
        print(f"  [warm] val_loss={mw['loss']:.4f} img F1={mw['img']['macro_f1']:.4f} | "
              f"subj {fmt(mw['subj'])}")

        set_trainable(model, CFG["backbone"], "head")
        opt = torch.optim.AdamW(param_groups(model, CFG["backbone"], V["lr_bb"],
                                             CFG["lr_head"]), weight_decay=CFG["weight_decay"])
        for e in range(FT_S1):
            print(f"  [s1 {e+1}] train_loss="
                  f"{run_epoch_diag(model, dl, crit, opt, scaler, V['bn_eval_all']):.4f}")
        m0 = eval_loader(model, va_dl, crit)
        print(f"  [s1 end] val_loss={m0['loss']:.4f} img F1={m0['img']['macro_f1']:.4f} | "
              f"subj {fmt(m0['subj'])}")

        set_trainable(model, CFG["backbone"], CFG["freeze_until"])
        snap = bn_snapshot(model)
        opt = torch.optim.AdamW(param_groups(model, CFG["backbone"], V["lr_bb"],
                                             CFG["lr_head_ft"]),
                                weight_decay=CFG["weight_decay"])
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_S2)
        best, best_e, wait = m0["subj"]["macro_f1"], "s1", 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        for e in range(FT_S2):
            l = run_epoch_diag(model, dl, crit, opt, scaler, V["bn_eval_all"])
            m = eval_loader(model, va_dl, crit); sch.step()
            print(f"  [s2 {e+1:2d}] tr={l:.4f} val={m['loss']:.4f} gap={m['loss']-l:+.3f} "
                  f"imgF1={m['img']['macro_f1']:.4f} "
                  f"imgAUC=[{','.join(f'{v:.3f}' for v in m['img']['auc'])}] "
                  f"subjF1={m['subj']['macro_f1']:.4f} subjBal={m['subj']['bal_acc']:.4f} "
                  f"subjAUC=[{','.join('nan' if np.isnan(v) else f'{v:.3f}' for v in m['subj']['auc'])}]")
            if m["subj"]["macro_f1"] > best:
                best, best_e, wait = m["subj"]["macro_f1"], f"s2e{e+1}", 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= FT_PATIENCE: print("  early stop"); break
        print(f"  best subjF1={best:.4f} @ {best_e}   ({time.time()-t:.0f}s)")
        print("  BN running-stat drift during stage 2 (relative L2, worst 6):")
        for n, dm_, dv in bn_drift(model, snap):
            print(f"    {n:30s} d_mean={dm_:.4f} d_var={dv:.4f}")
        model.load_state_dict(best_state)
        rr = eval_loader(model, va_dl, crit)
        rb = eval_loader(model, va_dl, crit, batchstats=True)
        dF1 = rb["subj"]["macro_f1"] - rr["subj"]["macro_f1"]
        print(f"  BEST weights, RUNNING BN stats : {fmt(rr['subj'])}")
        print(f"  BEST weights, BATCH   BN stats : {fmt(rb['subj'])}")
        print(f"  batch - running subjF1 = {dF1:+.4f} => BN running stats "
              f"{'ARE' if dF1 > 0.02 else 'are NOT'} the problem")
        print(f"  subject confusion (running stats):\n{np.array2string(rr['subj']['cm'])}")
        print(classification_report(rr["subj"]["true"], rr["subj"]["pred"],
                                    target_names=CLASS_NAMES, labels=[0, 1, 2],
                                    zero_division=0))
        logrow(test=f"ft_{V['tag']}", best=round(best, 4),
               auc=np.round(rr["subj"]["auc"], 3).tolist())
        del model; torch.cuda.empty_cache()

# ================================================================== MAIN ====
sys.stdout = _Tee(LOGPATH, _real_stdout)
try:
    for nm, fn in (("audit", b_audit), ("geometry", b_geometry), ("leakage", b_leakage),
                   ("permutation", b_permutation), ("probe_variants", b_probe_variants),
                   ("aggregation", b_aggregation), ("subtasks", b_subtasks),
                   ("slice_window", b_slice_window), ("train_ceiling", b_train_ceiling),
                   ("backbones", b_backbones), ("finetune", b_finetune)):
        block(nm, fn)
    banner("SUMMARY TABLE")
    print(pd.DataFrame(ROWS).to_string(index=False))
    print(f"\nTOTAL RUNTIME {time.time()-T0:.0f}s")
finally:
    sys.stdout.flush(); sys.stdout = _real_stdout
print("diagnostics done ->", LOGPATH)


## Notes / next steps
- `run_folds` is now all five folds by default. **One fold has ~56 subjects and only
  3-4 `mild_moderate` — a single subject flipping moves macro-F1 by ~0.05, so a
  single-fold number is noise. Never judge a change on one fold.**
- **Order of business:** run 5c first. It is seconds, and it tells you whether the
  expensive fine-tune in section 6 is earning its keep. Measured on the first full
  run: reference probe 0.58-0.61 vs fine-tune 0.375 — the fine-tune was losing.
- **Do not re-enable scale-changing augmentation** (`RandomResizedCrop`, zoom,
  `RandomErasing`) without re-running the `use_aug=True` sweep in 5c. See `build_tf`.
- Only evaluate the **test set** (`fold == -1`) once, after the recipe is frozen.
- **Early stopping is nested** (`nested_early_stop`): the epoch is chosen on a
  held-out 20% of the *training* subjects, and the reported fold is touched once.
  Under the old scheme, permuted labels scored macro-F1 0.394 vs a 0.289 baseline —
  ~0.10 of the headline was peeking. Expect the honest number to be lower and to be
  the one you can defend.
- Section 9's fine-tune block inherits `CFG['freeze_until']`; with `"head"` it trains
  nothing and its BN-drift readings are all zero. Set it locally to compare fine-tunes.
- Try `backbone = "resnet50"` as a second model and compare macro-F1.
- Add Grad-CAM on a few subjects for a sanity check against hidden leakage (research.md §9b).

### resnet18 parameter budget (3-class head)
| block | params | share |
|---|---:|---:|
| conv1 + bn1 | 9,536 | 0.1% |
| layer1 | 147,968 | 1.3% |
| layer2 | 525,568 | 4.7% |
| layer3 | 2,099,712 | 18.8% |
| layer4 | 8,393,728 | **75.1%** |
| fc head | 1,539 | 0.0% |
| **total** | **11,178,051** | |

Capacity is concentrated in `layer4`, so `freeze_until` is a coarse dial: `"layer3"`
still trains 94% of the net, `"layer4"` trains 75%, `"head"` trains 0.01%.

### Overfitting knobs (in `CFG`, roughly most → least effective)
| Knob | Now | If val F1 still falls while train loss drops |
|---|---|---|
| `central_lo/hi` | 124–136 | widen to 118–142 (more real data beats every other fix) |
| augmentation | RRC + affine + erasing | already aggressive; add mixup before loosening |
| `weight_decay` | 1e-2 | 3e-2 |
| `freeze_until` | `"layer4"` | `"head"` — a linear probe on ~3k images is a serious baseline, not a cop-out |
| `lr_backbone` | 2e-5 | 1e-5 |
| `dropout` | 0.5 | 0.6 |
| `label_smoothing` | 0.05 | 0.1 |

Stage 2 can now only *replace* the stage-1 model if it beats it on val macro-F1, so if
fine-tuning is hopeless on this data size you automatically keep the linear probe.

If instead **train loss stays high and val F1 plateaus low**, you've over-corrected —
loosen the other way (`freeze_until="layer3"`, `weight_decay=1e-3`).

### Checkpoint format changed
`model_fold{k}.pt` is now a dict, not a bare state_dict. To reload:

```python
ck = torch.load("model_fold0.pt", map_location="cpu", weights_only=False)
model = build_model(ck["cfg"]["backbone"])
model.load_state_dict(ck["state_dict"])
print(ck["val_macro_f1"], ck["best_epoch"])
```
